In [2]:
import pandas as pd
import numpy as np

# === 1) Load CSV ===
file_path = "Data/1/FVG_OB_2015-2025_No_Open_Fvg.csv"
df = pd.read_csv(file_path, sep=";")

# === 2) Column names ===
group_col = "rr_ext_less_0_5"
rr_col = "rr_ext"
profit_col = "profit"
win_col = "is_profitable"

# Check required columns
required_cols = [rr_col, profit_col, win_col]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in the file: {missing}")

# === 3) Create grouping column ===
df[group_col] = (pd.to_numeric(df[rr_col], errors="coerce") < 0.5).astype(int)

# === 4) Function to calculate group statistics ===
def calc_group_stats(subdf):
    trades = len(subdf)

    wins = subdf[subdf[win_col] == 1]
    losses = subdf[subdf[win_col] == 0]

    gross_profit = wins[profit_col].sum()
    gross_loss = losses[profit_col].sum()

    if gross_loss != 0:
        profit_factor = gross_profit / abs(gross_loss)
    elif gross_profit > 0:
        profit_factor = np.inf
    else:
        profit_factor = np.nan

    return pd.Series({
        "Trades": trades,
        "Wins": len(wins),
        "Losses": len(losses),
        "Win rate %": len(wins) / trades * 100 if trades else np.nan,
        "Sum profit": subdf[profit_col].sum(),
        "Avg profit": subdf[profit_col].mean(),
        "Median profit": subdf[profit_col].median(),
        "Avg win": wins[profit_col].mean() if len(wins) else np.nan,
        "Avg loss": losses[profit_col].mean() if len(losses) else np.nan,
        "Profit factor": profit_factor,
        "Avg rr_ext": subdf[rr_col].mean()
    })

# === 5) Build split table ===
result = (
    df.groupby(group_col, dropna=False)
      .apply(calc_group_stats)
      .reset_index()
      .sort_values(group_col)
)

# Share of total trades
result.insert(2, "Share %", result["Trades"] / len(df) * 100)

# === 6) Format output ===
result_display = result.copy()

float_cols = [
    "Share %",
    "Win rate %",
    "Sum profit",
    "Avg profit",
    "Median profit",
    "Avg win",
    "Avg loss",
    "Profit factor",
    "Avg rr_ext"
]

for col in float_cols:
    result_display[col] = result_display[col].astype(float).round(3)

int_cols = [group_col, "Trades", "Wins", "Losses"]

for col in int_cols:
    if result_display[col].isna().any():
        result_display[col] = result_display[col].astype("Int64")
    else:
        result_display[col] = result_display[col].astype(int)

display(
    result_display.style.format({
        group_col: "{:d}",
        "Trades": "{:d}",
        "Wins": "{:d}",
        "Losses": "{:d}",
        "Share %": "{:.2f}",
        "Win rate %": "{:.2f}",
        "Sum profit": "{:,.2f}",
        "Avg profit": "{:,.2f}",
        "Median profit": "{:,.2f}",
        "Avg win": "{:,.2f}",
        "Avg loss": "{:,.2f}",
        "Profit factor": "{:.3f}",
        "Avg rr_ext": "{:.3f}",
    })
)

,rr_ext_less_0_5,Trades,Share %,Wins,Losses,Win rate %,Sum profit,Avg profit,Median profit,Avg win,Avg loss,Profit factor,Avg rr_ext
0,0,115,70.12,55,60,47.83,"2,781.50",24.19,-999.02,"1,151.35","-1,009.04",1.046,1.337
1,1,49,29.88,41,8,83.67,"3,199.45",65.30,221.76,274.78,"-1,008.34",1.397,0.291
